# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [ ]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!pip install transformers -U

  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.2.3
    Uninstalling huggingface_hub-1.2.3:
      Successfully uninstalled huggingface_hub-1.2.3


In [ ]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.2.3 which is incompatible.


---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [ ]:
# batch_size와 epochs를 조정해보세요!
batch_size = 16
epochs = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 데이터셋 로드
raw_datasets = load_dataset("sst2")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [ ]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [ ]:
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc

# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 4210/4210 [24:15<00:00,  2.89it/s]


Epoch 1 - Avg Train Loss: 0.1831


Epoch 2: 100%|██████████| 4210/4210 [24:14<00:00,  2.90it/s]


Epoch 2 - Avg Train Loss: 0.1125
Validation Accuracy (google/electra-base-discriminator): 0.9450


## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명
2. 어떤 모델이 적합한지에 대한 본인의 의견
  - 학습 속도, accuracy 등 고려


# 1. 각 모델 구조 설명
---

## [BERT 모델]
- 트랜스포머 기반 양방향 인코더를 사용하는 자연어 처리 모델

**사전 학습**
- 다음 두 가지 태스크에 대해 사전학습 된다.

**1. 마스크 언어 모델링(MLM)**
- 주어진 문장에서 단어의 15%를 무작위로 마스킹한 후, 마스킹된 문장 전체를 모델에 입력하여 마스킹된 단어를 예측하는 방식.
- 일반적으로 단어를 순차적으로 입력받는 기존의 순환 신경망(RNN)이나, 내부적으로 미래의 토큰을 마스킹하는 GPT와 같은 자기회귀 모델과는 차별화됨.
- MLM을 통해 모델은 문장의 양방향 표현을 학습할 수 있다.

**2. 다음 문장 예측(NSP)**
- 모델은 사전 학습 과정에서 마스크 처리된 두 문장을 연결하여 입력으로 사용.
- 이 두 문장은 원문에서 서로 인접해 있던 문장인 경우도 있고, 그렇지 않은 경우도 있다.
- 모델은 두 문장이 서로 이어지는지 여부를 예측해야 함.


## [ELECTRA 모델]
- 트랜스포머 인코더 기반으로, BERT와 사전학습 방식이 다름.
- MLM에서 단어 예측: generator / 원본인지 아닌지 식별: Discriminator

**사전 학습**
- **replaced token detection(교체한 토큰 탐지)** 태스크에 대해 사전학습 된다.
 - MLM에서 [MLM] 토큰으로 마스킹할 대상이 되는 토큰들을 ‘다른 토큰’으로 변경한 뒤, (generator)
 - 이 토큰이 실제 토큰인지 아니면 교체한 토큰인지를 판별하는 형태로 학습을 진행한다. (Discriminator)



# 2. 어떤 모델이 더 적합한지 결과 분석
---

- Validation Accuracy (bert-base-uncased): 0.9071
- Validation Accuracy (google/electra-base-discriminator): 0.9450

## => ELECTRA 모델이 더 적합하다.
- 최종 검증 정확도만 놓고 봤을 때 ELECTRA가 BERT보다 약 3.8%p 더 높은 성능을 보임.
- 두 모델 모두 걸린 시간은 약 24분으로 같은 시간이 걸리므로, 동일 훈련 시간 대비 성능이 더 좋은 ELECTRA 모델이 더 학습 효율성이 높으며 언어의 문맥적 의미를 파악하는 능력이 우수하다고 판단할 수 있다.
- 또한 학습 과정에서는 BERT의 Train Loss가 ELECTRA보다 근소하게 더 낮았음에도 실제 검증 정확도는 ELECTRA가 더 높다. 이는 BERT의 사전 학습 방식에서 기인한 한계로 해석할 수 있다.
  - BERT는 사전 학습 때만 [MASK]라는 가짜 토큰을 보고, 지금처럼 실제 과제를 수행할 때는 이 토큰이 없어 사전 학습과 실제 추론 단계 간의 '불일치' 문제가 존재한다.
  - 하지만 ELECTRA는 실제 토큰이 올바른지 여부를 판별하는 방식으로 학습되기 때문에, 사전 학습과 실제 태스크 간의 입력 분포 차이가 상대적으로 작다. 이런 특성으로 인해 ELECTRA가 실제 검증 단계에서 더 안정적이고 높은 성능을 보인 것으로 판단된다.